## About the Data
- "The UK STATS19 dataset contains 100,000+ individual collision records per year with 38 columns including accident severity, speed limit, light conditions, pedestrian crossing details, and weather directly downloadable as CSV from the UK Department for Transport."
- The UK government records every single road accident reported to police. For each accident they write down things like - what was the weather, what time of the day, type of the road, how many vehicles, how many people involved. and how severe was it. Slight, Serious or Fatal.
- The STATS19 data is three separate tables - collisions, vehicles and casualties.


1. All three files share collision_index (primary key)
2. One collision can have 2 or more vehicles linked to it (one-to-many relationship)

In [91]:
import pandas as pd
import numpy as np

In [92]:
import os

print("Current Working Directory:")
print(os.getcwd())

print("\nFiles in current directory:")
print(os.listdir())

print("\nTrying data folder:")
print(os.listdir("../data/raw"))

Current Working Directory:
D:\Github\AccidentIQ\notebooks

Files in current directory:
['.ipynb_checkpoints', '1_data_loading.ipynb']

Trying data folder:
['.ipynb_checkpoints', 'dft-road-casualty-statistics-casualty-2022.csv', 'dft-road-casualty-statistics-collision-2022.csv', 'dft-road-casualty-statistics-vehicle-2022.csv']


In [93]:
collisions = pd.read_csv("../data/raw/dft-road-casualty-statistics-collision-2022.csv")
vehicles = pd.read_csv("../data/raw/dft-road-casualty-statistics-vehicle-2022.csv")
casualties = pd.read_csv("../data/raw/dft-road-casualty-statistics-casualty-2022.csv")

print("Collisions:", collisions.shape)
print("Vehicles:", vehicles.shape)
print("Casualties:", casualties.shape)

C:\Users\Anagha\AppData\Local\Temp\ipykernel_9984\2807274843.py:2: DtypeWarning: Columns (0: generic_make_model) have mixed types. Specify dtype option on import or set low_memory=False.
  vehicles = pd.read_csv("../data/raw/dft-road-casualty-statistics-vehicle-2022.csv")


Collisions: (106004, 44)
Vehicles: (193545, 32)
Casualties: (135480, 23)


In [94]:
print("\n Collision columns:")
for col in collisions.columns:
    print(" ", col)


 Collision columns:
  collision_index
  collision_year
  collision_ref_no
  location_easting_osgr
  location_northing_osgr
  longitude
  latitude
  police_force
  collision_severity
  number_of_vehicles
  number_of_casualties
  date
  day_of_week
  time
  local_authority_district
  local_authority_ons_district
  local_authority_highway
  local_authority_highway_current
  first_road_class
  first_road_number
  road_type
  speed_limit
  junction_detail_historic
  junction_detail
  junction_control
  second_road_class
  second_road_number
  pedestrian_crossing_human_control_historic
  pedestrian_crossing_physical_facilities_historic
  pedestrian_crossing
  light_conditions
  weather_conditions
  road_surface_conditions
  special_conditions_at_site
  carriageway_hazards_historic
  carriageway_hazards
  urban_or_rural_area
  did_police_officer_attend_scene_of_accident
  trunk_road_flag
  lsoa_of_accident_location
  enhanced_severity_collision
  collision_injury_based
  collision_adjusted_s

## Data Structure
The STATS19 2022 dataset contains three relational tables:
- collisions: 106004 x 44
- vehicles: 193545 x 32
- casualties: 135480 x 23

The 1:N relationship between collisions and vehicles or casualties means naive joining will inflate rows and cause data leakage.

In [95]:
# To decide which one becomes the target variable

severity_cols = [ "collisions_severity", "collision_injury_based", "enhanced_severity_collision" ]

for col in severity_cols:
    if col in collisions.columns:
        print({col})
        print(f"Missing values: {collisions[col].isna().sum()}")
        print(f"Unique values: {sorted(collisions[col].dropna().unique())}")
        counts = collisions[col].value_counts().sort_index()
        pct = collisions[col].value_counts(normalize=True).sort_index() * 100
        for val in counts.index:
            print(f" Value {val}: {counts[val]:,} rows ({pct[val]:.1f}%)")

{'collision_injury_based'}
Missing values: 0
Unique values: [np.int64(0), np.int64(1)]
 Value 0: 50,275 rows (47.4%)
 Value 1: 55,729 rows (52.6%)
{'enhanced_severity_collision'}
Missing values: 0
Unique values: [np.int64(-1), np.int64(1), np.int64(3), np.int64(5), np.int64(6), np.int64(7)]
 Value -1: 50,275 rows (47.4%)
 Value 1: 1,018 rows (1.0%)
 Value 3: 40,470 rows (38.2%)
 Value 5: 2,427 rows (2.3%)
 Value 6: 3,265 rows (3.1%)
 Value 7: 8,549 rows (8.1%)


In [96]:
# Check collision_severity directly and independently

print("Does column exist?", "collision_severity" in collisions.columns)
print()

print(collisions["collision_severity"].value_counts().sort_index())
print()
print("Missing values:", collisions["collision_severity"].isna().sum())
print("Unique values:", sorted(collisions["collision_severity"].unique()))

Does column exist? True

collision_severity
1     1602
2    23333
3    81069
Name: count, dtype: int64

Missing values: 0
Unique values: [np.int64(1), np.int64(2), np.int64(3)]


In [97]:
# Find every severity-related column
severity_related = [col for col in collisions.columns if "severity" in col.lower()]
print("All severity columns found:")
for col in severity_related:
    print(f"  {col}")

All severity columns found:
  collision_severity
  enhanced_severity_collision
  collision_adjusted_severity_serious
  collision_adjusted_severity_slight


## Target variable
collision_severity is our prediction target.

- ► collision_injury_based - binary 0/1, tells us which recording system was used, not severity
- ► enhanced_severity_collision - values like -1, 5, 6, 7 with no clear ordering, requires a data dictionary we don't have
- ► collision_severity - clean, no missing values, exactly 3 ordered classes, directly interpretable

In [98]:
# Choosing potential features

print("\n Collision columns:")
for col in collisions.columns:
    print(" ", col)


 Collision columns:
  collision_index
  collision_year
  collision_ref_no
  location_easting_osgr
  location_northing_osgr
  longitude
  latitude
  police_force
  collision_severity
  number_of_vehicles
  number_of_casualties
  date
  day_of_week
  time
  local_authority_district
  local_authority_ons_district
  local_authority_highway
  local_authority_highway_current
  first_road_class
  first_road_number
  road_type
  speed_limit
  junction_detail_historic
  junction_detail
  junction_control
  second_road_class
  second_road_number
  pedestrian_crossing_human_control_historic
  pedestrian_crossing_physical_facilities_historic
  pedestrian_crossing
  light_conditions
  weather_conditions
  road_surface_conditions
  special_conditions_at_site
  carriageway_hazards_historic
  carriageway_hazards
  urban_or_rural_area
  did_police_officer_attend_scene_of_accident
  trunk_road_flag
  lsoa_of_accident_location
  enhanced_severity_collision
  collision_injury_based
  collision_adjusted_s

In [99]:
potential_features = ["day_of_week", "time", "road_type", "speed_limit", "urban_or_rural_area",
                      "junction_detail", "junction_control", "light_conditions", "weather_conditions",
                      "road_surface_conditions", "carriageway_hazards", "number_of_vehicles", "number_of_casualties",]

print(f"Potential features selected: {len(potential_features)}")
print()

for col in potential_features:
    n_missing = collisions[col].isna().sum()
    n_unique = collisions[col].nunique()
    pct_missing = (n_missing / len(collisions)) * 100
    print(f"{col:45s} | unique: {n_unique:4d} | missing: {n_missing:5d} ({pct_missing:.1f}%)")

Potential features selected: 13

day_of_week                                   | unique:    7 | missing:     0 (0.0%)
time                                          | unique: 1440 | missing:     0 (0.0%)
road_type                                     | unique:    6 | missing:     0 (0.0%)
speed_limit                                   | unique:    6 | missing:     0 (0.0%)
urban_or_rural_area                           | unique:    3 | missing:     0 (0.0%)
junction_detail                               | unique:    7 | missing:     0 (0.0%)
junction_control                              | unique:    6 | missing:     0 (0.0%)
light_conditions                              | unique:    6 | missing:     0 (0.0%)
weather_conditions                            | unique:    9 | missing:     0 (0.0%)
road_surface_conditions                       | unique:    7 | missing:     0 (0.0%)
carriageway_hazards                           | unique:   14 | missing:     0 (0.0%)
number_of_vehicles              

In [100]:
collisions["weather_conditions"].value_counts()

weather_conditions
1    86939
2    10011
8     3295
9     2955
4      970
5      909
7      552
3      312
6       61
Name: count, dtype: int64

In [101]:
collisions["speed_limit"].value_counts().sort_index()

speed_limit
20    16452
30    57329
40     9162
50     4546
60    12599
70     5916
Name: count, dtype: int64

In [102]:
collisions["weather_conditions"].value_counts().sort_index()

weather_conditions
1    86939
2    10011
3      312
4      970
5      909
6       61
7      552
8     3295
9     2955
Name: count, dtype: int64

In [103]:
collisions["road_type"].value_counts().sort_index()

road_type
1     6285
2     2392
3    16127
6    76558
7     1887
9     2755
Name: count, dtype: int64

In [104]:
pd.crosstab(collisions["speed_limit"], 
            collisions["collision_severity"], 
            normalize="index") * 100

collision_severity,1,2,3
speed_limit,,,
20,0.407245,18.125456,81.467299
30,0.903557,20.968445,78.127998
40,1.724514,23.324602,74.950884
50,3.431588,25.890893,70.677519
60,3.905072,30.200810,65.894119
70,3.566599,20.469912,75.963489


## I] Speed limit vs Fatal Severity (Fatal (1), Serious (2), Slight (3))
As speed limit increases, the fatality rate increases monotonically.
But  there is an anomaly to be noticed: 70mph (motorways) slightly lower fatality than 60mph rural roads.
Hypothesis: motorways are better engineered despite higher speeds.
This suggests speed_limit x road_type interaction maybe meaningful.

► speed_limit is a strong potential feature.

In [105]:
pd.crosstab(
    collisions["light_conditions"],
    collisions["collision_severity"],
    normalize="index"
) * 100

collision_severity,1,2,3
light_conditions,,,
-1,100.000000,0.000000,0.000000
1,1.199300,21.379392,77.421308
4,1.540279,22.568040,75.891681
5,2.994792,25.781250,71.223958
6,5.382538,29.630963,64.986499
7,1.695906,16.491228,81.812865


## II] Light conditions vs Fatal severity
Fatality rate increases as lightning gets worse.

Anomaly 1: -1 (means missing data) shows 100% fatality - only 1 row.
Informative missingness - fatal accidents may have incomplete records.

Anomaly 2: Darkness increases fatality rate. and it gets worse as lightning gets worse. But value 7 - darkness with unknown lighting - shows only 1.69% fatality, which is lower than even daylight.
Hypothesis: unknown lighting may indicate unusual low speed locations.

► light_conditions is a strong feature.

In [106]:
for col in ["road_type", "junction_detail", "urban_or_rural_area"]:
    print(f"\n{col}:")
    print(pd.crosstab(collisions[col], collisions["collision_severity"], 
                      normalize="index").mul(100).round(1))


road_type:
collision_severity    1     2     3
road_type                          
1                   0.4  16.0  83.6
2                   0.5  15.8  83.7
3                   2.1  19.8  78.0
6                   1.6  23.7  74.7
7                   1.0  17.6  81.5
9                   0.1   9.3  90.6

junction_detail:
collision_severity    1     2     3
junction_detail                    
-1                  0.0   3.9  96.1
 0                  2.0  22.8  75.2
 13                 1.0  21.6  77.4
 16                 0.9  21.1  78.0
 17                 0.4  22.0  77.6
 18                 1.1  21.2  77.7
 19                 1.1  23.8  75.1

urban_or_rural_area:
collision_severity     1     2     3
urban_or_rural_area                 
1                    0.8  20.1  79.1
2                    3.0  26.0  71.0
3                    0.0  27.3  72.7


In [107]:
collisions["road_type"].value_counts().sort_index()

road_type
1     6285
2     2392
3    16127
6    76558
7     1887
9     2755
Name: count, dtype: int64

In [108]:
collisions[collisions["road_type"] == 9]["collision_severity"].value_counts().sort_index()

collision_severity
1       3
2     255
3    2497
Name: count, dtype: int64

In [109]:
mask_9 = collisions["road_type"] == 9

print("Road type 9: speed limit distribution:")
print(collisions[mask_9]["speed_limit"].value_counts().sort_index())

print("\nRoad type 9: urban or rural:")
print(collisions[mask_9]["urban_or_rural_area"].value_counts().sort_index())

Road type 9: speed limit distribution:
speed_limit
20    1316
30    1275
40      55
50      24
60      47
70      38
Name: count, dtype: int64

Road type 9: urban or rural:
urban_or_rural_area
1    2516
2     238
3       1
Name: count, dtype: int64


## III] Road Type 9 has unreliable fatality statistics

Road type 9: 2,755 rows, only 3 fatal accidents → 0.1% fatality rate.
This is not a meaningful pattern. 3 data points cannot represent 
true fatality risk for this road category.

Too sparse to analyze seperately. Will keep in dataset, tree models handle rare categories naturally.

In [110]:
for col in ["weather_conditions", "road_surface_conditions", 
            "urban_or_rural_area", "number_of_vehicles", 
            "number_of_casualties"]:
    print(f"\n{col}:")
    print(pd.crosstab(collisions[col], collisions["collision_severity"],
                      normalize="index").mul(100).round(1))


weather_conditions:
collision_severity    1     2     3
weather_conditions                 
1                   1.6  22.5  76.0
2                   1.4  22.0  76.5
3                   0.6  22.4  76.9
4                   3.7  25.2  71.1
5                   1.9  22.4  75.7
6                   0.0  16.4  83.6
7                   3.4  24.1  72.5
8                   0.9  17.8  81.4
9                   0.2  12.1  87.7

road_surface_conditions:
collision_severity         1     2     3
road_surface_conditions                 
-1                       0.1  13.7  86.2
 1                       1.4  22.0  76.6
 2                       1.9  23.0  75.0
 3                       1.3  25.1  73.6
 4                       1.5  19.4  79.1
 5                       0.8  19.8  79.4
 9                       0.2   8.8  91.0

urban_or_rural_area:
collision_severity     1     2     3
urban_or_rural_area                 
1                    0.8  20.1  79.1
2                    3.0  26.0  71.0
3                 

## Summary:
features : speed_limit, light_conditions, urban_or_rural_area, number_of_casualities, road_type, weather_conditions, road_surface_conditions, number_of_vehicles, junction_detail

In [111]:
print("Vehicles columns;")
for col in vehicles.columns:
    print("", col)

print("\nCasualities columns:")
for col in casualties.columns:
    print("",col)

Vehicles columns;
 collision_index
 collision_year
 collision_ref_no
 vehicle_reference
 vehicle_type
 towing_and_articulation
 vehicle_manoeuvre_historic
 vehicle_manoeuvre
 vehicle_direction_from
 vehicle_direction_to
 vehicle_location_restricted_lane_historic
 vehicle_location_restricted_lane
 junction_location
 skidding_and_overturning
 hit_object_in_carriageway
 vehicle_leaving_carriageway
 hit_object_off_carriageway
 first_point_of_impact
 vehicle_left_hand_drive
 journey_purpose_of_driver_historic
 journey_purpose_of_driver
 sex_of_driver
 age_of_driver
 age_band_of_driver
 engine_capacity_cc
 propulsion_code
 age_of_vehicle
 generic_make_model
 driver_imd_decile
 lsoa_of_driver
 escooter_flag
 driver_distance_banding

Casualities columns:
 collision_index
 collision_year
 collision_ref_no
 vehicle_reference
 casualty_reference
 casualty_class
 sex_of_casualty
 age_of_casualty
 age_band_of_casualty
 casualty_severity
 pedestrian_location
 pedestrian_movement
 car_passenger
 bus_

In [112]:
# most common vehicle type per collision
vehicle_type_per_collision = vehicles.groupby("collision_index")["vehicle_type"].agg(lambda x: x.mode()[0])

# did any vehicle skid
skidding_per_collision = vehicles.groupby("collision_index")["skidding_and_overturning"].max()

# average vehicle age
avg_vehicle_age = vehicles.groupby("collision_index")["age_of_vehicle"].mean()

# combine them
vehicles_agg = pd.DataFrame({
    "vehicle_type_mode": vehicle_type_per_collision,
    "any_skidding": skidding_per_collision,
    "avg_vehicle_age": avg_vehicle_age
}).reset_index()

print(vehicles_agg.shape)
print(vehicles_agg.head())

(106004, 4)
  collision_index  vehicle_type_mode  any_skidding  avg_vehicle_age
0   2022010352073                  3             0             2.50
1   2022010352573                  9             0            -1.00
2   2022010352575                  9             0             7.00
3   2022010352578                  9             0             7.00
4   2022010352580                  9             0             6.25


In [113]:
vehicles["age_of_vehicle"].value_counts().sort_index().head(10)

age_of_vehicle
-1    42929
 0     4820
 1     9526
 2     8234
 3    10094
 4    10251
 5    10479
 6    10641
 7    10107
 8     9251
Name: count, dtype: int64

In [114]:
# -1 means unknown, replace with mean
mean_age = vehicles[vehicles["age_of_vehicle"] != -1]["age_of_vehicle"].mean()
vehicles["age_of_vehicle"] = vehicles["age_of_vehicle"].replace(-1, mean_age)
print("Done. Mean age used:", round(mean_age, 1))

Done. Mean age used: 8.5


In [115]:
mean_age = vehicles[vehicles["age_of_vehicle"] != -1]["age_of_vehicle"].mean()
vehicles["age_of_vehicle"] = vehicles["age_of_vehicle"].replace(-1, mean_age)
print("Mean age used for imputation:", round(mean_age, 1))
print("Any -1 remaining:", (vehicles["age_of_vehicle"] == -1).sum())

Mean age used for imputation: 8.5
Any -1 remaining: 0


Replaced -1 vehicle ages with mean.
A better approach would be to add a binary flag for missingness.

In [116]:
mean_age = vehicles[vehicles["age_of_vehicle"] != -1]["age_of_vehicle"].mean()
print(f"Mean vehicle age (excluding unknowns): {mean_age:.1f} years")

# create indicator column before replacing

vehicles["vehicle_age_unknown"] = (vehicles["age_of_vehicle"] == -1).astype(int)

# now replace -1 with mean

vehicles["age_of_vehicle"] = vehicles["age_of_vehicle"].replace(-1, mean_age)

# verify

print(f"\nUnknown age flags: {vehicles['vehicle_age_unknown'].sum():,}")
print(f"Remaining -1 values: {(vehicles['age_of_vehicle'] == -1).sum()}")
print(f"\nVehicle age stats now:")
print(vehicles["age_of_vehicle"].describe().round(1))

Mean vehicle age (excluding unknowns): 8.5 years

Unknown age flags: 0
Remaining -1 values: 0

Vehicle age stats now:
count    193545.0
mean          8.5
std           5.1
min           0.0
25%           5.0
50%           8.5
75%          11.0
max         122.0
Name: age_of_vehicle, dtype: float64


► Max = 122.0?? A 122 year old vehicle could be a data entry error.

In [117]:
vehicles[vehicles["age_of_vehicle"] > 50]["age_of_vehicle"].value_counts().head(10)

age_of_vehicle
52.0    7
54.0    7
62.0    6
55.0    6
58.0    5
57.0    4
59.0    4
56.0    3
61.0    3
53.0    3
Name: count, dtype: int64

In [118]:
vehicles["age_of_vehicle"] = vehicles["age_of_vehicle"].clip(upper=50)

print("Max vehicle age now:", vehicles["age_of_vehicle"].max())

Max vehicle age now: 50.0


► Found vehicles with age > 50 years which is likely data entry errors or classic cars.
They are very few rows. Capped at 50 years to prevent outliers affecting the model.

In [119]:
worst_casualty = casualties.groupby("collision_index")["casualty_severity"].min()

any_pedestrian = casualties.groupby("collision_index")["casualty_type"].agg(
    lambda x: int((x == 0).any())
)

avg_casualty_age = casualties.groupby("collision_index")["age_of_casualty"].mean()

casualties_agg = pd.DataFrame({
    "worst_casualty_severity": worst_casualty,
    "any_pedestrian": any_pedestrian,
    "avg_casualty_age": avg_casualty_age
}).reset_index()

print(casualties_agg.shape)
print(casualties_agg.head())

(106004, 4)
  collision_index  worst_casualty_severity  any_pedestrian  avg_casualty_age
0   2022010352073                        3               0         17.000000
1   2022010352573                        3               0         42.000000
2   2022010352575                        3               0         20.000000
3   2022010352578                        3               0         45.500000
4   2022010352580                        3               0         42.666667


In [120]:

collisions_cols = [
    "collision_index",
    "collision_severity",
    "day_of_week",
    "time",
    "road_type",
    "speed_limit",
    "urban_or_rural_area",
    "junction_detail",
    "junction_control",
    "light_conditions",
    "weather_conditions",
    "road_surface_conditions",
    "carriageway_hazards",
    "number_of_vehicles",
    "number_of_casualities"
]

df = collisions[collision_cols].copy()

# join vehicles aggregation

df = df.merge(vehicles_agg, on="collision_index", how="left")

# join casualties aggregation

df = df.merge(casualties_agg, on="collision_index", how="left")

# drop collision_index 

df = df.drop(columns=["collision_index"])

print("Final dataset shape:", df.shape)
print("\n Missing values:")
print(df.isnull().sum())

Final dataset shape: (106004, 20)

 Missing values:
collision_severity         0
day_of_week                0
time                       0
road_type                  0
speed_limit                0
urban_or_rural_area        0
junction_detail            0
junction_control           0
light_conditions           0
weather_conditions         0
road_surface_conditions    0
carriageway_hazards        0
number_of_vehicles         0
number_of_casualties       0
vehicle_type_mode          0
any_skidding               0
avg_vehicle_age            0
worst_casualty_severity    0
any_pedestrian             0
avg_casualty_age           0
dtype: int64


In [122]:
import os
os.makedirs("../data/processed", exist_ok=True)
df.to_csv("../data/processed/accidents_clean.csv", index=False)
print("Saved.")

Saved.


## Final dataset
106004 rows x 20 columns
Zero missing values.

Built by 
- selecting 15 meaningful columns from the collisions table
- aggregating vehicles table to one row per collision (vehicle type, skidding, average age)
- aggregating the casualties table to one row per collision (worst severity, pedestrian flag, average age)
- joining all three on collision_index

This shows no row inflation, no data leakage, one clean row per accident.